In [29]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [30]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по КРС v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,КРС
1401,КОСТАНАЙСКАЯ ОБЛАСТЬ,2019-09-01,5036.22
1496,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2017-01-01,1462.89
240,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-06-01,7246.46
1345,КОСТАНАЙСКАЯ ОБЛАСТЬ,2015-01-01,3858.83
1011,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2018-12-01,14040.98
305,АЛМАТИНСКАЯ ОБЛАСТЬ,2019-04-01,6685.34
1443,КОСТАНАЙСКАЯ ОБЛАСТЬ,2023-03-01,2974.40
1579,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2023-12-01,3197.86
1094,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2015-04-01,3061.07
1646,МАНГИСТАУСКАЯ ОБЛАСТЬ,2018-12-01,560.61


In [31]:
# === загружаем данные ===
best_methods = pd.read_excel("results/КРС - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКТЮБИНСКАЯ ОБЛАСТЬ,4.65,243.58,6.46,362.00,8.93,462.49,HW,MAPE,4.65,243.58
1,АТЫРАУСКАЯ ОБЛАСТЬ,6.33,126.22,6.66,132.20,15.71,348.61,HW,MAPE,6.33,126.22
2,КАРАГАНДИНСКАЯ ОБЛАСТЬ,3.69,102.56,5.62,150.64,14.75,537.56,HW,MAPE,3.69,102.56
3,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,8.25,135.73,12.39,220.04,9.06,157.73,HW,MAPE,8.25,135.73
4,МАНГИСТАУСКАЯ ОБЛАСТЬ,26.67,48.95,43.21,20.85,55.99,99.49,HW,MAPE,26.67,20.85
5,ОБЛАСТЬ АБАЙ,13.01,581.70,13.21,548.97,32.95,1527.36,HW,MAPE,13.01,548.97
6,ОБЛАСТЬ ЖЕТІСУ,14.94,820.56,30.85,1753.38,31.17,1992.80,HW,MAPE,14.94,820.56
7,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,16.20,2420.04,16.74,2473.98,21.59,2945.10,HW,MAPE,16.20,2420.04
8,ГШЫМКЕНТ,35.63,164.10,37.61,178.60,33.20,164.59,Prophet,MAPE,33.20,164.10
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,4.54,188.09,4.25,194.85,4.02,162.47,Prophet,MAPE,4.02,162.47


In [ ]:
actual_aug = pd.read_excel("КРС 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["КРС"] = (actual_aug["КРС"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("КРС обработанные август 2025.xlsx", index=False)
actual_aug


In [33]:
# === настройки ===
TARGET = "КРС"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [34]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [35]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [37]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [38]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [39]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [ ]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/КРС - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

18:09:58 - cmdstanpy - INFO - Chain [1] start processing
18:09:58 - cmdstanpy - INFO - Chain [1] done processing
18:09:58 - cmdstanpy - INFO - Chain [1] start processing
18:09:58 - cmdstanpy - INFO - Chain [1] done processing
18:09:58 - cmdstanpy - INFO - Chain [1] start processing
18:09:59 - cmdstanpy - INFO - Chain [1] done processing
